# 04 Multivariate Stress

## Purpose

Build a joint-stress timeline and a stress-state index from synthetic multivariate series.

## Data / config used

- synthetic coupled logistic maps
- output directory: `outputs/notebooks/04_multivariate_stress/`


In [ ]:
from pathlib import Path

import pandas as pd

from dynsys_econometrics.multivariate import joint_exceedances, stress_state_index, wide_panel
from dynsys_econometrics.simulation import simulate_coupled_logistic_maps

output_dir = Path("outputs/notebooks/04_multivariate_stress")
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
values = simulate_coupled_logistic_maps(n_steps=500, n_series=3, coupling=0.15, seed=23)
dates = pd.date_range("2000-01-31", periods=500, freq="ME")
panel = pd.concat(
    [
        pd.DataFrame({"date": dates, "series_id": "stress_a", "value": values[:, 0]}),
        pd.DataFrame({"date": dates, "series_id": "stress_b", "value": values[:, 1]}),
        pd.DataFrame({"date": dates, "series_id": "stress_c", "value": values[:, 2]}),
    ],
    ignore_index=True,
)

joint = joint_exceedances(panel, quantile=0.95, min_active=2)
wide = wide_panel(panel)
thresholds = {column: float(wide[column].quantile(0.95)) for column in wide.columns}
mask = wide.ge(pd.Series(thresholds), axis=1).reset_index()
state = stress_state_index(mask)

joint.to_csv(output_dir / "joint_exceedances.csv", index=False)
state.to_csv(output_dir / "stress_state_index.csv", index=False)

joint.head()


## Limitations

Multivariate stress here is a synchronized-extremes diagnostic. It does not prove structural or causal linkage between the underlying variables.
